### PDAC and neural invasion - regions, niches and high vs low NI (mp_pdac02)

**Paper**: Chen MM, Gao Q, Ning H, Chen K, Gao Y, Yu M, Liu CQ, Zhou W, Pan J, Wei L, Dou W, Zhang D, Zhu L, Zhang Q, Chen R, Zhang Z. *Integrated single-cell and spatial transcriptomics uncover distinct cellular subtypes involved in neural invasion in pancreatic cancer.* Cancer Cell. 2025 Sep 8;43(9):1656-1676.e10. doi: [10.1016/j.ccell.2025.06.020](https://doi.org/10.1016/j.ccell.2025.06.020). PMID: 40680743.

Code: [github.com/PDAC-zhanglab/PDAC](https://github.com/PDAC-zhanglab/PDAC) (R figure scripts; copy in pdac/data/github_PDAC-zhanglab) - Data: GEO SuperSeries [GSE278694](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE278694) = [GSE278687](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE278687) (Visium, 21 sections) + [GSE278688](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE278688) (scRNA) + [GSE278689](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE278689) (FFPE snRNA) + GSE300435 (scTCR); raw reads GSA HRA008568.

It continues **mp_pdac01_prepare.ipynb**: the regions computed there (**pdac01_region_labels.tsv**: Leiden clusters of the cell-type shares of the 21 sections, named after the paper's 12 niches) are read, not recomputed, and studied as in the paper's Figures 1-3: which cell types define each niche, how niches and cell types differ between **low-NI** and **high-NI** tumours, the malignant subtypes and their diversity.

#### Libs: **src/libs**

- **single_cell_lib**

  - Class **SingleCell**: generic Scanpy, Squidpy, cell2location and LIANA methods

  - **DEFAULT_PARAMS**
- **single_cell_pdac_lib** (new)

  - Class **SingleCellPDAC(SingleCell)**: PDAC methods (sections, NI status and platforms, re-derived subclusters, RCTD / cell2location abundances, regions named after the paper's niches, sections as replicates for high vs low NI, nerve spots / invasion / rings)

  - **PDAC_PARAMS**, **PDAC_SUBCLUSTERS** (marker sets of the paper's subclusters), **PDAC_GENES** (26 PDAC / NI genes), **PDAC_NI_GENES**, **PDAC_NERVE_GENES**, **PAPER_NICHES**, **PDAC_FILES** (one file registry per notebook: 'pdac01' ... 'pdac04')

**Programs: src/programs**: pdac_build_h5ad.py (GEO -> h5ad), pdac_reference_subclusters.py (subclusters), run_rctd_pdac.R (RCTD), run_c2l_pdac.py (cell2location, GPU).

| section | content |
|---|---|
| 1 | spots, abundances and regions of mp_pdac01 |
| 2 | region table: size, sections, platforms, cell types, niche |
| 3 | Figure 1F: relative cell-type proportion in each niche (and region) |
| 4 | niches on the sections (Figure 1D right) |
| 5 | Figures 2E / 3D: niche share per section, low vs high NI (all sections; CytAssist only) |
| 6 | Figure 3C: PCA of the sections on niche shares |
| 7 | Figure 3B: TLS cell types and other cell types, low vs high NI |
| 8 | Figures 2C / 2J: malignant subtypes, pure spots, composition and Shannon diversity per section |
| 9 | co-occurrence of cell types across sections (Figure 3I analogue) |
| 10 | PDAC / NI genes per region (dot plot); NI-related gene score (Figure 2M on Visium) |
| 11 | interactive region maps |

<br>

Figures and tables have the prefix **pdac02_**.

### Critique of the analysis and how it is handled

1. **21 sections, 17 patients**: sections are the replicates (PA04, PA05 and PA11 have two sections each, so p-values are slightly optimistic); effect sizes and the per-section points are shown with every test.
2. **Platform confound**: all 5 low-NI sections are CytAssist, all FF / FFPE-v1 sections high-NI. Every high vs low test is repeated on the 12 CytAssist sections (5 low vs 7 high); a difference that disappears there is not interpretable as NI biology.
3. **Ridge plots of the paper** summarise the same 5 vs 16 values per niche; here box + points (marker = platform) show them directly.
4. **Niche names are inferred** from composition (assign_niches); the paper's 12 niches were assigned with the H&E and are not public. Malignant regions are named after their dominant subtype ('N07-10_Malignant (D04_Ductal-GABRP)'), since the paper's N07-N10 numbering is not published (except N10 = D02_Ductal-APOA2). 'mixed' regions have no clear niche; 'N00_unassigned' = spots RCTD could not deconvolve (< 100 UMIs).
5. **Deconvolution source**: RCTD (the paper's method) or cell2location (GPU map); the tables carry it in **abundance_source**.

## Loading packages

In [ ]:
import os, sys
from pathlib import Path
from importlib.metadata import version
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import gc

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42          # editable text in PDFs
%config InlineBackend.print_figure_kwargs = {'bbox_inches': 'tight', 'dpi': 90}   # inline display only; files are saved at 300 dpi
import plotly.io as pio
pio.renderers.default = "notebook_connected"

ROOT0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = ROOT0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.single_cell_pdac_lib import (SingleCellPDAC, PDAC_PARAMS, PDAC_SUBCLUSTERS, PDAC_GENES, PDAC_NI_GENES,
                                       PDAC_NERVE_GENES, PAPER_NICHES, PDAC_FILES)
from libs.single_cell_lib import cmap_byr, cmap_lbwr
from libs.matplotlib_figure_style import apply_figure_style
from libs.Basic import create_dir

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 200)
print({p: version(p) for p in ['scanpy', 'squidpy', 'anndata', 'plotly']})

## Parameters

In [ ]:
force=False       # True: recompute every table and redraw every figure (after changing parameters or inputs)
verbose=True

apply_figure_style()
scp = SingleCellPDAC(ROOT0, analysis='pdac02')
root_data, root_figure, root_result = scp.root_data, scp.root_figure, scp.root_result
print(scp)
scp.calc_df_files()

### 1. Spots, abundances and the regions of mp_pdac01

In [ ]:
df_regions = scp.read_tsv('region_labels', verbose=verbose)

source = df_regions['abundance_source'].iloc[0]
print("regions of mp_pdac01 made with:", source)

adata_vis = scp.read_h5ad('visium', set_self=False, verbose=verbose)
adata_vis = scp.remove_mt_genes(adata_vis)
if source == 'c2l':
    a_map = scp.read_h5ad_slots('c2l_map', skip=('layers',))
    adata_vis.obsm['q05_cell_abundance_w_sf'] = pd.DataFrame(a_map.obsm['q05_cell_abundance_w_sf'], index=a_map.obs_names).loc[adata_vis.obs_names]
    adata_vis.uns['mod'] = a_map.uns['mod']; del a_map

scp.calc_qc_metrics(adata_vis)
df_abund = scp.get_abundance(adata_vis, source=source)
df_share = scp.shares(df_abund)

d = df_regions.set_index('barcode').loc[adata_vis.obs_names]
for c in ('region_cluster', 'region_label', 'region_name', 'niche'):
    adata_vis.obs[c] = pd.Categorical(d[c].astype(str).values)

names = dict(zip(d.region_label.astype(str), d.region_name.astype(str)))
regions = list(adata_vis.obs['region_label'].cat.categories)
niches = sorted(adata_vis.obs['niche'].unique())
sections_rep = scp.representative_sections()
df_sections = scp.section_table(adata_vis)
print(adata_vis.shape, '|', len(regions), 'regions |', len(niches), 'niches |', source, df_abund.shape)

### 2. Region table

Per region: spots, sections, platforms, median UMIs, top cell types by share and by enrichment, the niche (mp_pdac01).

In [ ]:
df_region_summary = scp.calc_df_region_summary(adata_vis, df_share, names, fname='tsv_region_summary', force=force, verbose=verbose)
df_region_summary

### 3. Figure 1F: cell types in each niche

log2 enrichment of every cell type (share in the niche / share over all spots; clipped at 1/16). The paper's Figure 1F shows the scaled RCTD proportion per niche; the defining cell types should match: TLS (B, CD4/CD8 T), plasma, CAF, TAM, malignant ductal, non-malignant pancreas (acinar, D01), endocrine, peri-vascular, nerve (Schwann).

In [ ]:
force=False
verbose=False

df_enrich_niche = scp.plot_region_celltype_heatmap(adata_vis, df_share, key='niche', 
                                                   figsize=(20, 7), fontsize_scale=2,
                                                   fname=scp.fname('fig_niche_celltype'),
                                                   force=force, verbose=verbose)
plt.show()

In [ ]:
force=False
verbose=False

df_enrich = scp.plot_region_celltype_heatmap(adata_vis, df_share, key='region_label', 
                                             names=names, figsize=(20, 11), fontsize_scale=2,
                                             fname=scp.fname('fig_region_celltype'), force=force, verbose=verbose)
plt.show()

### 4. Niches on the sections

The paper's representative section PA98 (Figure 1D right) and the other representative sections.

In [ ]:
force=False
verbose=False

colors = scp.niche_colors(niches)          # the paper's niche colours (myUtiliity.R); malignant niches cycle its 4 malignant colours
fig = scp.plot_regions_on_sections(adata_vis, key='niche', sections=sections_rep, colors=colors, ncols=3, 
                                   panel_size=5, legend_ncol=3, bottom_scale=1,
                                   fname=scp.fname('fig_niches_sections'), force=force, verbose=verbose)
plt.show()

### 5. Figures 2E / 3D: niche share per section, low vs high NI

Share of each niche among the spots of a section; box + points per NI group (marker = platform), two-sided Mann-Whitney U over sections (BH FDR in the table). Paper: TLS, plasma-enriched and peri-vascular niches higher in low-NI tissues; Malignant_ductal_4 (D02_Ductal-APOA2) in low NI.

In [ ]:
df_niche_sec = scp.calc_df_share_by_section(adata_vis, 'niche', fname=scp.fname('tsv_niche_by_section'), force=force, verbose=verbose)
df_t_all = scp.calc_df_ni_test(df_niche_sec, fname='tsv_niche_ni_test', force=force, verbose=verbose)
df_t_cyt = scp.calc_df_ni_test(df_niche_sec, techniques=('CytAssist',), fname='tsv_niche_ni_test_cyt', force=force, verbose=verbose)
df_t_all.merge(df_t_cyt[['group', 'log2_ratio_high_low', 'p']], on='group', suffixes=('', '_CytAssist')).sort_values('p')

In [ ]:
import importlib

import libs.single_cell_lib as s   # the module where SingleCellPDAC is defined
importlib.reload(s)

import libs.single_cell_pdac_lib as m   # the module where SingleCellPDAC is defined
importlib.reload(m)

from libs.single_cell_pdac_lib import SingleCellPDAC

scp = SingleCellPDAC(ROOT0, analysis='pdac02')

In [ ]:
force=False
verbose=False

fig = scp.plot_ni_groups(df_niche_sec, niches, df_test=df_t_all, 
                         ncols=3, panel_size=(2.5, 3.5), figsize=(18,25), fontsize_scale=2,
                         title='Niche share per section (all 21 sections)',
                         fname=scp.fname('fig_niche_ni'), force=force, verbose=verbose)
plt.show()

In [ ]:
force=False
verbose=False

fig = scp.plot_ni_groups(df_niche_sec[df_niche_sec.technique == 'CytAssist'], niches, df_test=df_t_cyt, 
                         ncols=3, panel_size=(2.5, 3.5), figsize=(18,25), fontsize_scale=2,
                         title='Niche share per section (12 CytAssist sections: 5 low vs 7 high NI)',
                         fname=scp.fname('fig_niche_ni_cyt'), force=force, verbose=verbose)
plt.show()

### 6. Figure 3C: PCA of the sections on niche shares

Paper: the low-NI sections form a distinct cluster. Here colour = NI, marker = platform: separation along a platform axis would be technical.

In [ ]:
force=False
verbose=False

df_pca = scp.plot_section_pca(df_niche_sec, figsize=(11, 9), 
                              title='Sections on niche shares (Figure 3C analogue)',
                              fname=scp.fname('fig_section_pca'), force=force, verbose=verbose)
plt.show()

In [ ]:
df_pca.sort_values('PC1')

### 7. Figure 3B: cell types, low vs high NI

Mean share of each cell type per section. Paper (ST): B01_naive-TCL1A, B03_GCB-RGS13 and T10_CD4_Tfh-like-CXCL13 (TLS components) higher in low NI; here B cells and T cells are major types (lymphoid subclusters were not re-derived). Also shown: plasma, NLRP3+ TAM (M07), myCAF (F01), the malignant subtypes and Schwann subsets.

In [ ]:
force=False
verbose=False

df_ct_sec = scp.calc_df_celltype_by_section(adata_vis, df_abund, fname=scp.fname('tsv_celltype_by_section'), force=force, verbose=verbose)
df_ct_all = scp.calc_df_ni_test(df_ct_sec, group_col='cell_type', fname='tsv_celltype_ni_test', force=force, verbose=verbose)
df_ct_cyt = scp.calc_df_ni_test(df_ct_sec, group_col='cell_type', techniques=('CytAssist',), fname='tsv_celltype_ni_test_cyt', force=force, verbose=verbose)
df_ct_all.merge(df_ct_cyt[['cell_type', 'log2_ratio_high_low', 'p']], on='cell_type', suffixes=('', '_CytAssist')).sort_values('p').head(25)

In [ ]:
force=False
verbose=False

cts = ['B', 'CD4T', 'CD8T', 'Plasma', 'M07_Macro-NLRP3', 'M10_Macro-SPP1', 'F01_myCAF-MMP11', 'F02_Progenitor-CAF-DPT',
       'D02_Ductal-APOA2', 'D04_Ductal-GABRP', 'D09_Ductal-CEACAM6', 'S02_Schwann-TGFBI']
fig = scp.plot_ni_groups(df_ct_sec, cts, group_col='cell_type', df_test=df_ct_all, 
                         ncols=4, title='Cell-type share per section (all sections)',
                         fname=scp.fname('fig_celltype_ni'), force=force, verbose=verbose)
plt.show()

### 8. Malignant subtypes: pure spots (Figure 2C), composition and diversity (Figure 2J)

**Pure spots**: share of one malignant subtype >= 0.8 of the spot (the paper's criterion on its RCTD proportions); shown on the H&E of the section with most such spots. **Composition**: malignant subtypes (D02-D09) as a share of the malignant cells of each section, with the Shannon index. Paper: lower diversity in high-NI tissues; D02_Ductal-APOA2 in low NI.

In [ ]:
force=False
verbose=False

mal = [c for c in df_share.columns if c[:3] in ('D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09')]
pure = (df_share[mal] >= 0.8)
print(pure.sum().to_string())
for sub in ['D08_Ductal-CEACAM5', 'D09_Ductal-CEACAM6', 'D04_Ductal-GABRP', 'D02_Ductal-APOA2']:
    if sub in pure and pure[sub].any():
        lib = adata_vis.obs.loc[pure[sub].values, 'section'].value_counts().index[0]
        adata_vis.obs['_pure'] = np.where(pure[sub].values, sub, '')
        fig = scp.plot_regions_on_sections(adata_vis, key='_pure', sections=[lib], colors={sub: '#FFEE00'}, ncols=1, panel_size=6,
                                           fname=scp.fname('fig_malignant_pure', sub=sub.split('_')[0]), force=force, verbose=verbose)
        plt.show()

In [ ]:
force=False
verbose=False

df_div = scp.calc_df_malignant_diversity(adata_vis, df_abund, fname='tsv_malignant_diversity', force=force, verbose=verbose)
fig = scp.plot_malignant_diversity(df_div, fname='fig_malignant_comp', force=force, verbose=verbose)
plt.show()

In [ ]:
scp.calc_df_diversity_test(df_div)

### 9. Co-occurrence of cell types across sections (Figure 3I analogue)

Spearman correlation of the mean cell-type shares over the 21 sections; pairs with rho > 0.6 (the paper's threshold; the paper used sc/sn proportions per sample). Paper: M07_Macro-NLRP3 with T11_CD4_Th17-like (not separated here).

In [ ]:
force=False
verbose=False

R = scp.calc_df_celltype_correlation(df_ct_sec)
pairs = scp.calc_df_cooccurrence(R, min_rho=0.6, fname='tsv_cooccurrence', force=force, verbose=verbose)

fig = scp.plot_cooccurrence(R, fname='fig_cooccurrence', force=force, verbose=verbose)
plt.show()

In [ ]:
pairs.head(30)

### 10. PDAC / NI genes per region; NI-related gene score

Dot plot of the 24 genes of **PDAC_GENES** per region (fraction of spots > 0; mean log1p(CP10k) / max over the regions). **NI score** (Figure 2M on the spots): mean z (within section) of the 34 NI-related genes of the paper (**PDAC_NI_GENES**; those measured by the three panels), averaged per region.

In [ ]:
force=False
verbose=False

order = [r for r in regions]
df_frac, df_mean, df_rel = scp.calc_df_dotplot(list(PDAC_GENES), 'region_label', adata_vis[:, adata_vis.var.in_all_panels.values], order=order)

fig = scp.plot_dotplot(df_frac, df_rel, row_labels={r: f"{r} {names.get(r, '')}" for r in order}, cmap='RdPu', 
                       figsize=(17, 14), fontsize_scale=2, 
                       title='PDAC / NI genes per region', fname=scp.fname('fig_region_genes'), force=force, verbose=verbose)
plt.show()

In [ ]:
ni_score = scp.nerve_marker_score(adata_vis[:, adata_vis.var.in_all_panels.values], genes=PDAC_NI_GENES)
adata_vis.obs['NI_gene_score'] = ni_score
df_ni = adata_vis.obs.groupby('region_label', observed=True).NI_gene_score.agg(['mean', 'median', 'size']).assign(niche=lambda d: d.index.map(names))
df_ni.sort_values('mean', ascending=False).round(3)

### 11. Interactive region maps

Hover: niche, top cell types. One HTML per representative section in **pdac/figures** (**pdac02_region_map_<section>.html**); PA98 is shown.

In [ ]:
force=False
verbose=False

hover = {r: f"{row.top_share}" for r, row in df_region_summary.set_index('region_label').iterrows()}
lib = scp.params['library_main']

fig = scp.plotly_spatial_visium(names, adata_vis, key='region_label', library_id=lib, reset_button=True,
                                hover_extra=hover, fontsize_title=20, fontsize_legend=12, fontsize_hover=13, 
                                marker_line_color="#222222", fname='fig_region_map',
                                force=force, verbose=verbose)
if fig:
    fig.show()

### Notes for interpretation

- **Confound**: read the CytAssist-only columns before the all-section ones; the platform changes depth, gene panel and the deconvolution.

- **Sections, not patients**: 21 sections of 17 patients (two sections for PA04, PA05, PA11).
- **TLS**: B / T cells are major types here; the TLS niche is defined by co-occurring B and T cells, not by GC B or Tfh states.
- **Malignant niches** are numbered by dominant subtype, not by the paper's (unpublished) assignment.